In [1]:
import re, os, glob, shutil, subprocess
import numpy as np
import pandas as pd
from functools import reduce
from pyscf import gto, scf, lo, dft
from pyscf.tools import cubegen
import py3Dmol

np.set_printoptions(formatter={'float': '{: 0.3f}'.format})

/home/lector/python-QChem/lib/python3.9/site-packages/pyscf/dft/libxc.py:772: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, the same to the B3LYP functional in Gaussian and ORCA (issue 1480). To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


In [2]:
mol_xyz = """6
Methanol from DOI:10.5517/cc9m4qr
    O        1.04741      1.49087      8.41073
    H        1.74445      1.34154      8.85497
    C        1.17799      2.72349      7.72855
    H        1.27976      3.44847      8.38036
    H        0.37779      2.88268      7.18523
    H        1.96657      2.69343      7.14726
"""
mol = gto.M(atom='\n'.join(mol_xyz.split('\n')[2:]), basis='def2-svp', verbose=0)
mf = dft.RKS(mol, xc='B3LYP').run()


Mulliken

In [3]:
pop_mull, q_mull = mf.mulliken_pop()

 ** Mulliken pop  **
pop of  0 O 1s            1.98820
pop of  0 O 2s            1.05587
pop of  0 O 3s            0.61837
pop of  0 O 2px           1.00428
pop of  0 O 2py           0.92336
pop of  0 O 2pz           1.02307
pop of  0 O 3px           0.51435
pop of  0 O 3py           0.55964
pop of  0 O 3pz           0.60080
pop of  0 O 3dxy          0.00223
pop of  0 O 3dyz          0.00208
pop of  0 O 3dz^2         0.00129
pop of  0 O 3dxz          0.00451
pop of  0 O 3dx2-y2       0.00549
pop of  1 H 1s            0.77523
pop of  1 H 2s            0.04742
pop of  1 H 2px           0.03108
pop of  1 H 2py           0.02343
pop of  1 H 2pz           0.02922
pop of  2 C 1s            1.98493
pop of  2 C 2s            0.84275
pop of  2 C 3s           -0.04948
pop of  2 C 2px           0.80987
pop of  2 C 2py           0.64889
pop of  2 C 2pz           0.76102
pop of  2 C 3px           0.24350
pop of  2 C 3py           0.15846
pop of  2 C 3pz           0.21162
pop of  2 C 3dxy          0

IAO

In [4]:
occ_coeff = mf.mo_coeff[:, mf.mo_occ > 0]

iao = lo.iao.iao(mol, occ_coeff)
iao = lo.vec_lowdin(iao, mf.get_ovlp())
ibo = lo.ibo.ibo(mol, occ_coeff, iaos=iao)

pmol = mol.copy()
pmol.build(False, False, basis='minao')

S = mf.get_ovlp()
c_iao  = reduce(np.dot, (iao.T, S, occ_coeff))
dm_iao = 2 * np.dot(c_iao, c_iao.T)
pop_iao, q_iao = mf.mulliken_pop(pmol, dm_iao, s=np.eye(pmol.nao_nr()))

 Iterative localization: IB/P4/2x2, 7 iter; Final gradient 1.96e-09
 ** Mulliken pop  **
pop of  0 O 1s            1.99570
pop of  0 O 2s            1.55970
pop of  0 O 2px           1.74190
pop of  0 O 2py           1.55806
pop of  0 O 2pz           1.72591
pop of  1 H 1s            0.63077
pop of  2 C 1s            1.98743
pop of  2 C 2s            0.98429
pop of  2 C 2px           1.23007
pop of  2 C 2py           0.92053
pop of  2 C 2pz           1.13730
pop of  3 H 1s            0.85112
pop of  4 H 1s            0.82582
pop of  5 H 1s            0.85139
 ** Mulliken atomic charges  **
charge of    0O =     -0.58128
charge of    1H =      0.36923
charge of    2C =     -0.25962
charge of    3H =      0.14888
charge of    4H =      0.17418
charge of    5H =      0.14861


NPA

In [5]:

C = lo.orth_ao(mf, 'nao')
mo = np.linalg.solve(C, mf.mo_coeff)

dm = mf.make_rdm1(mo, mf.mo_occ)
mf.mulliken_pop(mol, dm, np.eye(mol.nao_nr()))

 ** Mulliken pop  **
pop of  0 O 1s            1.99996
pop of  0 O 2s            1.63084
pop of  0 O 3s            0.00327
pop of  0 O 2px           1.70607
pop of  0 O 2py           1.60226
pop of  0 O 2pz           1.73701
pop of  0 O 3px           0.00459
pop of  0 O 3py           0.00191
pop of  0 O 3pz           0.00423
pop of  0 O 3dxy          0.00135
pop of  0 O 3dyz          0.00157
pop of  0 O 3dz^2         0.00074
pop of  0 O 3dxz          0.00292
pop of  0 O 3dx2-y2       0.00325
pop of  1 H 1s            0.57799
pop of  1 H 2s            0.00365
pop of  1 H 2px           0.00149
pop of  1 H 2py           0.00107
pop of  1 H 2pz           0.00152
pop of  2 C 1s            1.99996
pop of  2 C 2s            1.00481
pop of  2 C 3s            0.00057
pop of  2 C 2px           1.20704
pop of  2 C 2py           0.87001
pop of  2 C 2pz           1.10415
pop of  2 C 3px           0.00066
pop of  2 C 3py           0.00099
pop of  2 C 3pz           0.00028
pop of  2 C 3dxy          0

(array([ 2.000,  1.631,  0.003,  1.706,  1.602,  1.737,  0.005,  0.002,
         0.004,  0.001,  0.002,  0.001,  0.003,  0.003,  0.578,  0.004,
         0.001,  0.001,  0.002,  2.000,  1.005,  0.001,  1.207,  0.870,
         1.104,  0.001,  0.001,  0.000,  0.001,  0.003,  0.001,  0.002,
         0.002,  0.840,  0.002,  0.000,  0.002,  0.002,  0.821,  0.001,
         0.002,  0.000,  0.001,  0.840,  0.002,  0.002,  0.000,  0.001]),
 array([-0.700,  0.414, -0.197,  0.154,  0.174,  0.154]))

In [6]:
C_ibo_in_iao = iao.T @ S @ ibo
def make_l_list(mol):
    shell_list = np.zeros(mol.nao)
    i = 0
    for j in range(len(mol.ao_loc_nr())-1):
        l = mol.bas_angular(j)
        k = 0
        while k < 2*l+1:
            shell_list[i+k] = l
            k+=1
        i+=2*l+1
    return shell_list     

def analyze_ibo_hybridization_per_atom(ibo_vec, mol, pmol):
    aoslice = pmol.aoslice_by_atom()
    l_list = make_l_list(pmol)
    
    results = {}

    for ia in range(mol.natm):
        start_ao, stop_ao = aoslice[ia][2:4]
        contrib_vec = ibo_vec[start_ao:stop_ao] ** 2        
        contrib = contrib_vec.sum()
        
        lab = pmol.ao_labels()[start_ao:stop_ao]
        for i in range(len(lab)):
            lab[i] = lab[i].strip().split()[-1]
        
        s_pct = p_pct = d_pct = f_pct = 0.0
        
        for i in range(len(contrib_vec)):
            if l_list[start_ao+i-1] == 0 :
                s_pct+=100*contrib_vec[i]/contrib
            elif l_list[start_ao+i-1] == 1 :
                p_pct+=100*contrib_vec[i]/contrib
            elif l_list[start_ao+i-1] == 2 :
                d_pct+=100*contrib_vec[i]/contrib
            elif l_list[start_ao+i-1] == 3 :
                f_pct+=100*contrib_vec[i]/contrib
                
        results[ia] = {
                'atom': mol.atom_symbol(ia)+str(ia),
                'norm_contrib': float(contrib),
                'vec' : contrib_vec,
                'labels' : lab,
                's': s_pct,
                'p': p_pct,
                'd': d_pct,
                'f': f_pct}  
    return results

In [7]:
print("Анализ гибридизации в IBO:")
for i in range(C_ibo_in_iao.shape[1]):
    hyb = analyze_ibo_hybridization_per_atom(C_ibo_in_iao[:, i], mol, pmol)
    if not hyb:
        continue
        
    print(f"\nIBO {i+1}:")
    for ia, data in hyb.items():
        if data['norm_contrib'] > 0.02:
            print(f"  Атом {data['atom']} ({ia}): "
                  f"вклад = {data['norm_contrib']:.2f}"
                  f"  s = {data['s']:.1f}%, p = {data['p']:.1f}%, d = {data['d']:.1f}% "
                  f"\n        {data['labels']}"
                  f"\n        {data['vec']}")

Анализ гибридизации в IBO:

IBO 1:
  Атом O0 (0): вклад = 1.00  s = 100.0%, p = 0.0%, d = 0.0% 
        ['1s', '2s', '2px', '2py', '2pz']
        [ 0.991  0.009  0.000  0.000  0.000]

IBO 2:
  Атом C2 (2): вклад = 1.00  s = 98.7%, p = 1.3%, d = 0.0% 
        ['1s', '2s', '2px', '2py', '2pz']
        [ 0.987  0.012  0.000  0.001  0.000]

IBO 3:
  Атом O0 (0): вклад = 0.68  s = 36.9%, p = 63.1%, d = 0.0% 
        ['1s', '2s', '2px', '2py', '2pz']
        [ 0.003  0.250  0.266  0.030  0.135]
  Атом H1 (1): вклад = 0.31  s = 100.0%, p = 0.0%, d = 0.0% 
        ['1s']
        [ 0.314]

IBO 4:
  Атом C2 (2): вклад = 0.58  s = 25.7%, p = 74.3%, d = 0.0% 
        ['1s', '2s', '2px', '2py', '2pz']
        [ 0.002  0.137  0.291  0.003  0.147]
  Атом H5 (5): вклад = 0.42  s = 100.0%, p = 0.0%, d = 0.0% 
        ['1s']
        [ 0.419]

IBO 5:
  Атом O0 (0): вклад = 0.63  s = 26.8%, p = 73.2%, d = 0.0% 
        ['1s', '2s', '2px', '2py', '2pz']
        [ 0.001  0.167  0.001  0.342  0.115]
  Атом C

In [8]:
cubegen.orbital(mol, "IBO_1.cube", ibo[:, 4])

array([[[-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        [-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        [-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        ...,
        [ 0.000,  0.000,  0.000, ...,  0.000,  0.000,  0.000],
        [ 0.000,  0.000,  0.000, ...,  0.000,  0.000,  0.000],
        [ 0.000,  0.000,  0.000, ...,  0.000,  0.000,  0.000]],

       [[-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        [-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        [-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        ...,
        [ 0.000,  0.000,  0.000, ...,  0.000,  0.000,  0.000],
        [ 0.000,  0.000,  0.000, ...,  0.000,  0.000,  0.000],
        [ 0.000,  0.000,  0.000, ...,  0.000,  0.000,  0.000]],

       [[-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        [-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        [-0.000, -0.000, -0.000, ...,  0.000,  0.000,  0.000],
        ...,
        [ 0.

mo_data = open('IBO_1.cube', 'r').read()
view = py3Dmol.view(width=600, height=250)
view.addModel(mol_xyz, 'xyz')
view.setStyle({"stick":{"radius":0.15},"sphere": {"scale":0.25}})
view.addVolumetricData(mo_data, 'cube',
                       {'isoval': -0.01, 'color': "red", 'opacity': 0.9})
view.addVolumetricData(mo_data, 'cube',
                       {'isoval': 0.01, 'color': "blue", 'opacity': 0.9})
view.show()

In [9]:
dm = mf.make_rdm1()
_ = cubegen.density(mol, "density.cube", dm, nx=100, ny=100, nz=100) 
cri_in = """## Read the molecular geometry from the cube file.
molecule density.cube

## Read the electron density from the cube file.
load density.cube

## Locate the critical points.
auto 


## Integrate density over atomic basins.
bader

## Write bond paths assiciated with BCPs to file.
fluxprint
graph 2
text
endfluxprint
"""
open('cri.in', 'w').write(cri_in)

crit = shutil.which('cricri') or os.path.expanduser('~/bin/cricri')
with open('cri.in') as fin, open('cri.out', 'w') as fout:
    subprocess.run([crit], stdin=fin, stdout=fout, check=True)

In [10]:
lines = open('cri.out').read().splitlines()

def read_table(lines, header_must_have, ncols, first=True):
    for k, ln in enumerate(lines):
        if ln.lstrip().startswith('#') and all(t in ln for t in header_must_have):
            rows = []
            for ln2 in lines[k + 1:]:
                t = ln2.split()
                if not t or t[0].startswith('-') or t[0].startswith('#') or t[0] == 'Sum':
                    break
                if len(t) >= ncols:
                    rows.append(t[:ncols])
            if rows:
                return rows
    raise ValueError(f'таблица с заголовком {header_must_have} не найдена в cri.out')

# критические точки
cp = pd.DataFrame(read_table(lines, ['ncp', 'type', 'CPname'], 10),
                  columns=['ncp', 'type', 'CPname', 'x', 'y', 'z', 'name', 'rho', '|grad|', 'lap'])
for col in ['ncp', 'x', 'y', 'z', 'rho', '|grad|', 'lap']:
    cp[col] = cp[col].astype(float)
cp['ncp'] = cp['ncp'].astype(int)

# связевые точки
bp = pd.DataFrame(read_table(lines, ['End-1', 'End-2'], 9),
                  columns=['ncp', 'Elem_A1', 'N_A1', 'Elem_A2', 'N_A2', 'r1', 'r2', 'r1/r2', 'A1-B-A2'])
for col in ['ncp', 'r1', 'r2', 'r1/r2', 'A1-B-A2']:
    bp[col] = bp[col].astype(float)
bp['ncp'] = bp['ncp'].astype(int)

# интегрирование по бассейнам
ch = pd.DataFrame(read_table(lines, ['Id', 'cp', 'ncp', 'Name', 'Pop'], 9),
                  columns=['Id', 'cp', 'ncp', 'Name', 'Z', 'mult', 'Volume', 'Pop', 'Lap'])
for col in ['Id', 'Z', 'Volume', 'Pop', 'Lap']:
    ch[col] = ch[col].astype(float)
ch[['Id', 'Z']] = ch[['Id', 'Z']].astype(int)

print('Критические точки (Å; rho и Лапласиан в a.u.):')
display(cp)
print('Связевые пути , Å):')
display(bp)
print('Интегрированные атомные свойства:')
display(ch)

Критические точки (положения в Å; rho и Лапласиан в ат. ед.):


,ncp,type,CPname,x,y,z,name,rho,|grad|,lap
0,1,"(3,-3)",nucleus,1.047410,1.490870,8.410730,O,141.525079,0.000000e+00,-71622.871500
1,2,"(3,-3)",nucleus,1.744450,1.341540,8.854970,H,0.465062,0.000000e+00,-14.194835
2,3,"(3,-3)",nucleus,1.177990,2.723490,7.728550,C,68.776028,0.000000e+00,-26796.616900
3,4,"(3,-3)",nucleus,1.279760,3.448470,8.380360,H,0.441823,0.000000e+00,-14.519077
4,5,"(3,-3)",nucleus,0.377790,2.882680,7.185230,H,0.436635,0.000000e+00,-16.010432
5,6,"(3,-3)",nucleus,1.966570,2.693430,7.147260,H,0.442304,0.000000e+00,-16.268528
6,7,"(3,-1)",bond,0.664850,2.821883,7.382839,b3,0.353005,4.070467e-16,-1.606036
7,8,"(3,-1)",bond,1.242642,3.176571,8.139291,b2,0.352678,7.841835e-16,-1.677488
8,9,"(3,-1)",bond,1.673382,2.702191,7.365439,b4,0.352597,1.639845e-13,-1.650996
9,10,"(3,-1)",bond,1.145013,2.296244,7.971438,b1,0.255976,1.932809e-16,-0.401670


Связевые пути (длины r1, r2 — от BCP до атомов, Å):


,ncp,Elem_A1,N_A1,Elem_A2,N_A2,r1,r2,r1/r2,A1-B-A2
0,7,C,(3),H,(5),0.6265,0.3538,1.770977,178.9702
1,8,C,(3),H,(4),0.6150,0.3653,1.683573,179.3384
2,9,H,(6),C,(3),0.3656,0.6146,0.594815,179.2602
3,10,O,(1),C,(3),0.9226,0.4926,1.872984,177.5506


Интегрированные атомные свойства:


,Id,cp,ncp,Name,Z,mult,Volume,Pop,Lap
0,1,1,1,O,8,--,267.774878,9.225638,0.029676
1,2,2,2,H,1,--,58.541655,0.286238,-0.059380
2,3,3,3,C,6,--,87.334818,5.356668,0.125412
3,4,4,4,H,1,--,164.580578,1.056422,-0.051845
4,5,5,5,H,1,--,135.447325,1.002298,-0.024307
5,6,6,6,H,1,--,140.818699,1.052414,-0.019556


In [11]:
ch['q_QTAIM'] = ch['Z'] - ch['Pop']
print('QTAIM atomic charges:\n')
for _, r in ch.iterrows():
    print(f"{r['Name']}{int(r['Id'])-1}   {r['q_QTAIM']: .4f}     (N = {r['Pop']:.4f}, V = {r['Volume']:.1f} bohr^3)")
print('\nСумма зарядов:', round(ch['q_QTAIM'].sum(), 4), '  Σ населённостей:', round(ch['Pop'].sum(), 4), '(должно быть ≈', mol.nelectron, ')')

QTAIM atomic charges:

O0   -1.2256     (N = 9.2256, V = 267.8 bohr^3)
H1    0.7138     (N = 0.2862, V = 58.5 bohr^3)
C2    0.6433     (N = 5.3567, V = 87.3 bohr^3)
H3   -0.0564     (N = 1.0564, V = 164.6 bohr^3)
H4   -0.0023     (N = 1.0023, V = 135.4 bohr^3)
H5   -0.0524     (N = 1.0524, V = 140.8 bohr^3)

Сумма зарядов: 0.0203   Σ населённостей: 17.9797 (должно быть ≈ 18 )
